<a href="https://colab.research.google.com/github/iamtrask/abcGPT/blob/main/notebooks/train_dual_alt_mixed_beta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# abcGPT dual-source — alt_mixed + Beta_half

Slim training notebook for the `alt_mixed` + `beta_half` variant. Same alt_mixed mode (mixed batches every iter, two optimizers, pass-level alternation of which slot steps), but alpha is sampled from Beta(0.5, 0.5) instead of Uniform. This concentrates training at the alpha=0,1 corners so each slot's gradient signal during its training window is mostly "improve as a standalone model."

**Just the training**, no analysis prelude. Hit Run All.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install --quiet zstandard tiktoken

In [ ]:
import os
if not os.path.exists('/content/abcGPT'):
    !git clone --depth 1 https://github.com/iamtrask/abcGPT.git /content/abcGPT
else:
    !cd /content/abcGPT && git pull --rebase --autostash
%cd /content/abcGPT

In [ ]:
!python data/shakespeare_wiki_char/prepare.py

## Run directory (tagged `-altmixed-beta` so this run is distinct from the `-altmixed` Uniform run)

In [ ]:
RUN_ID = None   # set explicitly to resume, e.g. "20260519-XXXXXX-altmixed-beta"

In [ ]:
import time, os, glob
DRIVE_ROOT = '/content/drive/MyDrive/abcGPT/runs'
os.makedirs(DRIVE_ROOT, exist_ok=True)
if 'RUN_ID' not in dir() or RUN_ID is None:
    RUN_ID = time.strftime('%Y%m%d-%H%M%S') + '-altmixed-beta'
elif not RUN_ID.endswith('-altmixed-beta'):
    RUN_ID = RUN_ID + '-altmixed-beta'
OUT_DIR_DUAL = f'{DRIVE_ROOT}/{RUN_ID}'
os.makedirs(OUT_DIR_DUAL, exist_ok=True)
print('run dir:', OUT_DIR_DUAL)
snaps = sorted(glob.glob(os.path.join(OUT_DIR_DUAL, '*.pt.zst')))
print(f'  {len(snaps)} existing snapshots' + (' (will resume)' if snaps else ' (fresh run)'))

## Train

In [ ]:
!python train_dual.py config/train_shakespeare_wiki_dual_alt_mixed_beta.py \
    --out_dir=$OUT_DIR_DUAL \
    --batch_mode=alt_mixed \
    --first_pass_corpus=shake \
    --mix_distribution=beta_half